# 02 - Silver Transformation
Clean, deduplicate and enrich retail sales data.

In [ ]:
from pyspark.sql import functions as F
base_path = '/mnt/retail'
sales = spark.read.format('delta').load(f'{base_path}/bronze/sales')
products = spark.read.format('delta').load(f'{base_path}/bronze/products')

In [ ]:
silver_sales = (
    sales.dropDuplicates(['transaction_id'])
         .filter(F.col('quantity') > 0)
         .withColumn('sale_date', F.to_date('sale_date'))
         .withColumn('unit_price', F.col('unit_price').cast('double'))
         .withColumn('total_amount', F.col('quantity') * F.col('unit_price'))
)

silver_sales = silver_sales.join(
    products.select('product_code', 'product_name', 'category'),
    'product_code', 'left'
)

In [ ]:
silver_sales.write.format('delta').mode('overwrite').partitionBy('sale_date').save(f'{base_path}/silver/sales')